# NB2 experiment: 30% systematic label noise and early stopping

**COMP3620 CW3**

## The question

Does early stopping still protect a network from label noise when the mistakes are systematic rather than random?

## What NB2 does

NB1 reproduced the known result: under 30% random noise, the network learns the true digits first and memorises the wrong labels later, so early stopping keeps most of the test accuracy.

NB2 changes exactly one thing, the **type** of noise. Each selected image is relabelled as the next digit, `c → (c + 1) mod 10` (pair flipping, Han et al., 2018). The noise ratio, the selected images, the data split, the model and the training settings are all the same as in NB1.

NB2 covers:

- the theory and pre-registered predictions P5 and P6, committed before any result is generated;
- 30% systematic noise and a check of the corrupted labels;
- learning-rate tuning for the systematic condition, with at least three seeds;
- a comparison with the clean and random results from NB1;
- tests of P5 and P6, and analyses that rule out alternative explanations;
- a sensitivity check;
- scope, limitations and next steps.

NB2 reads NB1's cached results from `results/`, so NB1 does not need to be run first.


## Theory and pre-registered predictions

*Written and committed before any systematic run was trained.*

### Why systematic noise should be harder

Take every training image whose true digit is 3 and look at the labels it was given:

| | labelled 3 | labelled 4 | labelled each other digit |
|---|---|---|---|
| random noise | 70% | ≈ 3.3% | ≈ 3.3% |
| systematic noise | 70% | 30% | 0% |

Early in training, a network learns the label that is most common for each kind of image. Under random noise the true label leads by about 21 to 1, and the wrong labels point in different directions, so their gradients largely cancel (coherent gradients, Chatterjee, 2020). They can only be memorised one image at a time, late in training.

Under systematic noise the true label still wins, but only by 7 to 3, and every wrong label points the same way. "c → c + 1" then behaves like a second rule that the network can learn. We therefore expect predictions to be pulled towards the next digit early in training, before the early-stopping epoch, and not only during late memorisation.

### Can a noisy validation set still choose the stopping epoch?

We stop at the epoch with the highest noisy-validation accuracy, because clean validation labels are not available in practice. Let a be the true validation accuracy and b the fraction of corrupted validation images predicted as the next digit. Then

  noisy validation accuracy ≈ a(1 − η) + bη.

Under random noise b stays small, so the noisy accuracy follows a (NB1, P4). Under systematic noise b can grow. An example with 1,000 validation images, 700 correctly labelled and 300 flipped:

| epoch | true accuracy a | where the errors go | noisy validation accuracy |
|---|---|---|---|
| A | 90% | spread over all digits | (630 + 3) / 1000 = 63.3% |
| B | 88% | 90% of errors on the next digit | (616 + 32) / 1000 = 64.8% |

The noisy validation set prefers B, although A is better on the true labels. We do not predict whether this happens in our runs; it is analysed separately after P5 and P6.

### Predictions

**P5, main hypothesis.** With the same 30% noise ratio, early stopping protects clearly less under systematic noise than under random noise.

**P6.** Under systematic noise, test errors concentrate on the next digit. Under random noise and clean labels, the share of test errors on the next digit stays near 1/9.

### Metrics and thresholds, fixed now

The primary metric is the same as in NB1: test accuracy at the noisy-validation early-stopping epoch, as a mean over seeds 0 to 2, each condition at its own selected learning rate.

Baseline from NB1's cached results, known before writing these thresholds: random early-stopping test accuracy 93.41% (seeds 92.85%, 93.47%, 93.91%); share of test errors on the next digit at the final epoch, clean 0.097 and random 0.109.

- **P5 holds if both are true:**
  - the mean early-stopping test accuracy under systematic noise is at least **2 percentage points** below the random mean;
  - every systematic seed is below the lowest random seed.
- **P6 holds if all are true**, using `test_err_next` at each run's early-stopping epoch, mean over seeds:
  - systematic ≥ **0.50**;
  - random < **0.20**;
  - clean < **0.20**.


In [21]:
QUICK = True  # True: load the cached runs and finish in < 5 min; False: run the full experiment

## Shared protocol, from the group worksheet

| Item | Setting |
|---|---|
| Dataset | MNIST, `torchvision.datasets.MNIST` |
| Train | `train=True` images `[0:10000]`, 10,000 |
| Validation | `train=True` images `[50000:55000]`, 5,000 |
| Test | `train=False` images, 10,000; never corrupted |
| Input | pixels scaled to 0 to 1, flattened to 784 |
| Model | MLP 784 → 512 → 512 → 10, ReLU, PyTorch default init |
| Regularisation | none: no dropout, weight decay, BatchNorm, or data augmentation |
| Loss / optimiser | cross-entropy; Adam with default betas/eps, no weight decay |
| Batch size | 128, reshuffled every epoch |
| Epochs | 200, no early stopping during training; evaluate after every epoch |
| Early stopping | after training: epoch with highest **noisy** validation accuracy |
| Noise | η = 0.30, 30% of each class, rounded to the nearest whole image |
| Seeds | 0, 1, 2, plus 3 and 4 if time |
| Learning rates | 1e-4, 3e-4, 1e-3, 3e-3 |


In [22]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

ROOT = Path(".")
DATA_DIR = ROOT / "data"
RESULTS_DIR = ROOT / "results"
FIG_DIR = ROOT / "figures"
for p in (DATA_DIR, RESULTS_DIR, FIG_DIR):
    p.mkdir(parents=True, exist_ok=True)

NOISE_RATIO = 0.30 # 30% of each class is corrupted
NUM_CLASSES = 10
EPOCHS = 200       # train to the end
BATCH_SIZE = 128
HIDDEN = 512
LR_CANDIDATES = [1e-4, 3e-4, 1e-3, 3e-3]
SEEDS = [0, 1, 2]
CONDITIONS_NB1 = ["clean", "random"]  # conditons of NB1, NB2 would be systemetic
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("device:", DEVICE)
print("torch:", torch.__version__)
print("results dir:", RESULTS_DIR.resolve())


device: cpu
torch: 2.15.0
results dir: /Users/alexw/Desktop/3620project/results


In [23]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


## 1. Data and fixed split

The split is fixed once and reused by every seed and every condition. It does not depend on the model or on the noise type.

| Part | Source | Size | Labels |
|---|---|---|---|
| Train | `train=True` images `[0:10000]` | 10,000 | changed according to condition |
| Validation | `train=True` images `[50000:55000]` | 5,000 | changed in the same way; clean copy kept for analysis |
| Test | `train=False` images | 10,000 | never changed |


In [24]:
def load_mnist_split():
    tf = transforms.ToTensor()
    train_full = datasets.MNIST(DATA_DIR, train=True, download=True, transform=tf)
    test_full = datasets.MNIST(DATA_DIR, train=False, download=True, transform=tf)

    train_idx = list(range(0, 10_000))
    val_idx = list(range(50_000, 55_000))

    def to_tensors(dataset):
        xs, ys = [], []
        for x, y in dataset:
            xs.append(x.reshape(-1))
            ys.append(int(y))
        return torch.stack(xs), torch.tensor(ys, dtype=torch.long)

    train_x, train_y = to_tensors(Subset(train_full, train_idx))
    val_x, val_y = to_tensors(Subset(train_full, val_idx))
    test_x, test_y = to_tensors(test_full)
    return train_x, train_y, val_x, val_y, test_x, test_y

train_x, train_y, val_x, val_y, test_x, test_y = load_mnist_split()
data = (train_x, train_y, val_x, val_y, test_x, test_y)

print("train:", tuple(train_x.shape), tuple(train_y.shape), "counts:", np.bincount(train_y.numpy()))
print("val:  ", tuple(val_x.shape), tuple(val_y.shape), "counts:", np.bincount(val_y.numpy()))
print("test: ", tuple(test_x.shape), tuple(test_y.shape), "counts:", np.bincount(test_y.numpy()))
print("pixel range:", float(train_x.min()), float(train_x.max()))


train: (10000, 784) (10000,) counts: [1001 1127  991 1032  980  863 1014 1070  944  978]
val:   (5000, 784) (5000,) counts: [502 534 497 521 484 457 485 527 515 478]
test:  (10000, 784) (10000,) counts: [ 980 1135 1032 1010  982  892  958 1028  974 1009]
pixel range: 0.0 1.0


## 2. Label noise

- **Random, symmetric:** each selected image is assigned a label drawn uniformly from the other 9 classes; the assigned label is always wrong.
- **Systematic, pair flipping, NB2:** each selected image is assigned the label `c + 1` modulo 10.
- **Clean:** labels unchanged, but the same mask is returned for analysis.

The mask is generated first from `np.random.default_rng`, seeded with the pair `seed` and `split_id`, so it is identical for clean/random/systematic. Training uses `split_id=0`; validation uses `split_id=1`; the test set is never touched.

We use integer labels, so no one-hot regeneration is needed. If the lab machine has no internet, pre-download MNIST into `data/` before running the notebook.

Because the fixed 10k/5k splits are not exactly balanced across classes, the number selected per class is rounded to the nearest image. The realised ratio therefore differs from 30% by at most 0.5 image per class; the check below prints the actual per-class ratios.


In [25]:
def make_mask(labels, ratio, rng):
    labels = np.asarray(labels)
    mask = np.zeros(len(labels), dtype=bool)
    for c in range(NUM_CLASSES):
        idx = np.flatnonzero(labels == c)
        n = int(round(len(idx) * ratio))
        if n > 0:
            mask[rng.choice(idx, size=n, replace=False)] = True
    return mask


def corrupt_labels(labels, noise_type, ratio, seed, split_id):
    labels = np.asarray(labels)
    rng = np.random.default_rng([seed, split_id])
    # Generate the mask first: it is identical for clean/random/systematic.
    mask = make_mask(labels, ratio, rng)
    corrupted = labels.copy()

    if noise_type == "random" and mask.any():
        true = labels[mask]
        wrong = rng.integers(0, NUM_CLASSES - 1, size=len(true))
        wrong = np.where(wrong >= true, wrong + 1, wrong)  # uniformly skip the true class
        corrupted[mask] = wrong
    elif noise_type == "systematic" and mask.any():
        corrupted[mask] = (labels[mask] + 1) % NUM_CLASSES
    elif noise_type == "clean":
        pass
    else:
        raise ValueError(f"unknown noise_type: {noise_type}")

    return corrupted, mask


def transition_table(true_labels, given_labels, mask=None, only_masked=False):
    """10x10 table of true digit against given label, row-normalised.

    By default every image of a class is counted, which is the orientation the
    shared protocol specifies: under 30% random noise the diagonal is 0.70 and
    each of the other nine cells is about 0.033; under paired flipping the
    diagonal is 0.70 and the "next digit" cell is 0.30.

    only_masked=True restricts the table to the corrupted images, giving diagonal
    0 and 1/9 per off-diagonal cell under random noise. That answers a different
    question, namely what a corrupted image was corrupted to, and is kept as a
    secondary view.
    """
    true_labels = np.asarray(true_labels)
    given_labels = np.asarray(given_labels)
    if mask is not None and only_masked:
        true_labels = true_labels[mask]
        given_labels = given_labels[mask]
    table = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
    for t, g in zip(true_labels, given_labels):
        table[t, g] += 1
    row_sums = table.sum(axis=1, keepdims=True)
    return table, table / np.maximum(row_sums, 1)


def check_transition(labels, corrupted, mask, noise_type, n_sigma=4.5):
    """Check a noise table against the shape the shared protocol predicts.

    Tolerances are derived from the actual class counts instead of being hard-coded.
    The diagonal is exact by construction, because the mask corrupts round(r * n_c)
    images of class c, so the diagonal is 1 - round(r * n_c) / n_c, and it gets a
    tight tolerance. An off-diagonal cell holds about r * n_c / 9 corrupted images
    and is therefore a binomial count, with normalised standard deviation
    sqrt((r/9)/n_c); the tolerance is n_sigma of that. A fixed tolerance would
    either fail at random on the 5,000-image validation split or be too loose to
    detect a real error.
    """
    labels = np.asarray(labels)
    _, row_norm = transition_table(labels, corrupted)
    diag = np.diag(row_norm)

    counts = np.bincount(labels, minlength=NUM_CLASSES)
    n_min = max(int(counts[counts > 0].min()), 1)
    diag_tol = 0.5 / n_min + 0.002
    cell_tol = n_sigma * float(np.sqrt((NOISE_RATIO / (NUM_CLASSES - 1)) / n_min))

    if noise_type == "systematic":
        next_digit = np.array([row_norm[c, (c + 1) % NUM_CLASSES] for c in range(NUM_CLASSES)])
        others = row_norm.copy()
        for c in range(NUM_CLASSES):
            others[c, c] = 0.0
            others[c, (c + 1) % NUM_CLASSES] = 0.0
        ok = (
            np.all(np.abs(diag - (1 - NOISE_RATIO)) <= diag_tol)
            and np.all(np.abs(next_digit - NOISE_RATIO) <= cell_tol)
            and np.all(others <= cell_tol)
        )
        detail = (f"diagonal {diag.min():.3f} to {diag.max():.3f}, want {1 - NOISE_RATIO:.2f} +/-{diag_tol:.3f}, "
                  f"next digit {next_digit.min():.3f} to {next_digit.max():.3f}, want {NOISE_RATIO:.2f} +/-{cell_tol:.3f}, "
                  f"all other cells <= {others.max():.3f}")
        return ok, row_norm, detail

    off = row_norm[~np.eye(NUM_CLASSES, dtype=bool)]
    target = NOISE_RATIO / (NUM_CLASSES - 1)
    ok = (
        np.all(np.abs(diag - (1 - NOISE_RATIO)) <= diag_tol)
        and np.all(np.abs(off - target) <= cell_tol)
    )
    detail = (f"diagonal {diag.min():.3f} to {diag.max():.3f}, want {1 - NOISE_RATIO:.2f} +/-{diag_tol:.3f}, "
              f"off-diagonal {off.min():.3f} to {off.max():.3f}, want {target:.3f} +/-{cell_tol:.3f}")
    return ok, row_norm, detail


def check_noise(labels, seed, split_id, name):
    labels = np.asarray(labels)
    print(f"\n=== {name} | seed {seed} ===")
    masks = {}
    for noise_type in ["clean", "random", "systematic"]:
        corrupted, mask = corrupt_labels(labels, noise_type, NOISE_RATIO, seed, split_id)
        masks[noise_type] = mask
        ratios = [
            ((labels == c) & mask).sum() / max((labels == c).sum(), 1)
            for c in range(NUM_CLASSES)
        ]
        print(f"{noise_type:10s} per-class ratio: min={min(ratios):.4f} max={max(ratios):.4f}")
        if noise_type in ("random", "systematic"):
            ok, row_norm, detail = check_transition(labels, corrupted, mask, noise_type)
            print("  full transition table, rows = true digit, cols = given label:")
            print(np.round(row_norm, 3))
            print(f"  check: {detail}  -> {'PASS' if ok else 'FAIL'}")
            _, masked_norm = transition_table(labels, corrupted, mask, only_masked=True)
            print("  restricted to corrupted images only, secondary view:")
            print(np.round(masked_norm, 3))

    same = (
        np.array_equal(masks["clean"], masks["random"])
        and np.array_equal(masks["clean"], masks["systematic"])
    )
    print("same mask across noise types:", same)


check_noise(train_y.numpy(), seed=0, split_id=0, name="train")
check_noise(val_y.numpy(), seed=0, split_id=1, name="val")

print("\nCompact check across all seeds:")
for s in SEEDS:
    for split_id, labels, name in [(0, train_y.numpy(), "train"), (1, val_y.numpy(), "val")]:
        masks = {}
        ratios_ok = True
        for nt in ["clean", "random", "systematic"]:
            _, m = corrupt_labels(labels, nt, NOISE_RATIO, s, split_id)
            masks[nt] = m
            ratios = [((labels == c) & m).sum() / max((labels == c).sum(), 1) for c in range(NUM_CLASSES)]
            # Counts are rounded to the nearest image, so the realised ratio can differ
            # from 0.30 by at most 0.5 image per class.
            ratios_ok = ratios_ok and all(
                abs(r - NOISE_RATIO) <= 0.5 / max((labels == c).sum(), 1) + 1e-12
                for c, r in enumerate(ratios)
            )
        same = (
            np.array_equal(masks["clean"], masks["random"])
            and np.array_equal(masks["clean"], masks["systematic"])
        )
        print(f"seed {s} {name}: ratios_ok={ratios_ok}, same_mask={same}")



=== train | seed 0 ===
clean      per-class ratio: min=0.2996 max=0.3004
random     per-class ratio: min=0.2996 max=0.3004
  full transition table, rows = true digit, cols = given label:
[[0.7   0.037 0.042 0.024 0.03  0.023 0.036 0.032 0.039 0.037]
 [0.035 0.7   0.03  0.043 0.036 0.027 0.032 0.028 0.035 0.034]
 [0.031 0.03  0.7   0.037 0.039 0.035 0.025 0.034 0.038 0.028]
 [0.044 0.021 0.037 0.7   0.036 0.021 0.04  0.041 0.031 0.03 ]
 [0.037 0.037 0.037 0.03  0.7   0.037 0.027 0.033 0.032 0.033]
 [0.03  0.028 0.034 0.031 0.035 0.7   0.032 0.049 0.028 0.034]
 [0.039 0.035 0.033 0.036 0.033 0.036 0.7   0.04  0.029 0.019]
 [0.034 0.039 0.033 0.033 0.029 0.036 0.034 0.7   0.038 0.024]
 [0.036 0.033 0.03  0.034 0.03  0.035 0.032 0.034 0.7   0.037]
 [0.027 0.04  0.036 0.024 0.029 0.039 0.03  0.037 0.04  0.7  ]]
  check: diagonal 0.700 to 0.700, want 0.70 +/-0.003, off-diagonal 0.019 to 0.049, want 0.033 +/-0.028  -> PASS
  restricted to corrupted images only, secondary view:
[[0.    0.123 

## 3. Model and training protocol

Design choices and rationale:

- **10,000 training images.** Small enough that a high-capacity MLP can memorise the noisy labels within 200 epochs, and large enough that the clean pattern is still learnable.
- **784 → 512 → 512 → 10 MLP.** Enough capacity to fit the training set; PyTorch default initialisation; ReLU.
- **No regularisation.** No dropout, weight decay, BatchNorm, or data augmentation. These would slow memorisation and make it unclear whether a difference is due to the noise type or to regularisation. This is also a scope limitation.
- **Adam, default betas/eps, no weight decay; batch size 128; 200 epochs.** Fixed for every seed and condition. No early stopping during training, so the full curve is available; the stopping epoch is chosen afterwards from the noisy validation curve.
- **Cross-entropy**, integer labels. The test set is evaluated after every epoch but is never used for tuning or stopping.

The function `train_one_run` below is the shared training function. **NB2 must copy this whole training block** `set_seed`, `corrupt_labels`, `MLP`, `predict`, `compute_metrics`, `train_one_run` verbatim, so that the only difference is the noise type.

### What the seed controls

Per the shared protocol the seed fixes three channels and nothing else:

- **which images are corrupted and what they are corrupted to**, from `np.random.default_rng` with the pair `seed` and `split_id`, where `split_id=0` is train and `split_id=1` is validation. The mask depends on the seed only, so all three noise types corrupt the same images;
- **weight initialisation**, from `torch.manual_seed` inside `set_seed`;
- **batch order**, from a separate `torch.Generator` seeded with the same seed.

The split, the test set and the dropout masks are not affected: the split is fixed in code, the test set is never touched, and this model has no dropout. Because the mask is a function of the seed, the three seeds are three independent noise realisations, so the spread in section 5 covers both initialisation noise and noise-realisation noise.


In [26]:
class MLP(nn.Module):
    def __init__(self, hidden=HIDDEN, num_classes=NUM_CLASSES):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, num_classes),
        )

    def forward(self, x):
        return self.net(x)


@torch.inference_mode()
def predict(model, loader):
    model.eval()
    preds = []
    for x, _ in loader:
        preds.append(model(x.to(DEVICE)).argmax(dim=1).cpu())
    return torch.cat(preds).numpy()


def compute_metrics(model, train_loader, val_loader, test_loader,
                    train_y_clean, train_y_given, train_mask,
                    val_y_clean, val_y_given, test_y, condition):
    train_pred = predict(model, train_loader)
    val_pred = predict(model, val_loader)
    test_pred = predict(model, test_loader)

    train_acc_given = float((train_pred == train_y_given).mean())

    if train_mask.any():
        selected_pred_true = float((train_pred[train_mask] == train_y_clean[train_mask]).mean())
        nonselected_pred_true = (
            float((train_pred[~train_mask] == train_y_clean[~train_mask]).mean())
            if (~train_mask).any() else None
        )
        selected_pred_given = (
            None if condition == "clean"
            else float((train_pred[train_mask] == train_y_given[train_mask]).mean())
        )
    else:
        selected_pred_true = None
        nonselected_pred_true = None
        selected_pred_given = None

    val_acc_noisy = float((val_pred == val_y_given).mean())
    val_acc_clean = float((val_pred == val_y_clean).mean())
    test_acc = float((test_pred == test_y).mean())

    errors = test_pred != test_y
    if errors.any():
        next_digit = (test_y + 1) % NUM_CLASSES
        test_err_next = float((test_pred[errors] == next_digit[errors]).mean())
    else:
        test_err_next = None

    return {
        "epoch": None,
        "train_acc_given": train_acc_given,
        "selected_pred_true": selected_pred_true,
        "selected_pred_given": selected_pred_given,
        "nonselected_pred_true": nonselected_pred_true,
        "val_acc_noisy": val_acc_noisy,
        "val_acc_clean": val_acc_clean,
        "test_acc": test_acc,
        "test_err_next": test_err_next,
    }


def train_one_run(condition, lr, seed, data, epochs=EPOCHS, batch_size=BATCH_SIZE,
                  demo=False, verbose=True):
    train_x, train_y_clean_t, val_x, val_y_clean_t, test_x, test_y_t = data
    set_seed(seed)

    train_y_clean = train_y_clean_t.numpy()
    val_y_clean = val_y_clean_t.numpy()
    test_y = test_y_t.numpy()

    train_y_given, train_mask = corrupt_labels(
        train_y_clean, condition, NOISE_RATIO, seed, split_id=0
    )
    val_y_given, _ = corrupt_labels(
        val_y_clean, condition, NOISE_RATIO, seed, split_id=1
    )

    if demo:
        n_train = min(2000, len(train_x))
        n_val = min(1000, len(val_x))
        n_test = min(1000, len(test_x))
        train_x = train_x[:n_train]
        train_y_clean = train_y_clean[:n_train]
        train_y_given = train_y_given[:n_train]
        train_mask = train_mask[:n_train]
        val_x = val_x[:n_val]
        val_y_clean = val_y_clean[:n_val]
        val_y_given = val_y_given[:n_val]
        test_x = test_x[:n_test]
        test_y = test_y[:n_test]

    train_ds = TensorDataset(train_x, torch.tensor(train_y_given, dtype=torch.long))
    val_ds = TensorDataset(val_x, torch.tensor(val_y_given, dtype=torch.long))
    test_ds = TensorDataset(test_x, torch.tensor(test_y, dtype=torch.long))

    g = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, generator=g)
    train_eval_loader = DataLoader(train_ds, batch_size=512, shuffle=False)
    val_loader = DataLoader(val_ds, batch_size=512, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=512, shuffle=False)

    model = MLP().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)  # default betas/eps, no weight decay
    loss_fn = nn.CrossEntropyLoss()

    history = []
    t0 = time.perf_counter()

    for epoch in range(1, epochs + 1):
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = loss_fn(model(x), y)
            loss.backward()
            optimizer.step()

        metrics = compute_metrics(
            model, train_eval_loader, val_loader, test_loader,
            train_y_clean, train_y_given, train_mask,
            val_y_clean, val_y_given, test_y, condition,
        )
        metrics["epoch"] = epoch
        history.append(metrics)

        if verbose and (epoch % 20 == 0 or epoch == epochs):
            print(
                f"  epoch {epoch:3d}/{epochs} | train {metrics['train_acc_given']:.3f} "
                f"| val noisy {metrics['val_acc_noisy']:.3f} | test {metrics['test_acc']:.3f}"
            )

    val_noisy = np.array([h["val_acc_noisy"] for h in history])
    test_acc = np.array([h["test_acc"] for h in history])
    best_epoch = int(np.argmax(val_noisy)) + 1
    optimal_test_epoch = int(np.argmax(test_acc)) + 1

    metrics_dict = {key: [h[key] for h in history] for key in history[0]}

    return {
        "config": {
            "condition": condition,
            "lr": float(lr),
            "seed": int(seed),
            "epochs": int(epochs),
            "batch_size": int(batch_size),
            "noise_ratio": float(NOISE_RATIO),
            "noise_type": condition,
            "num_classes": int(NUM_CLASSES),
            "hidden": int(HIDDEN),
            "model": "MLP 784-512-512-10, ReLU, PyTorch default init",
            "regularisation": "none: no dropout, weight decay, BatchNorm, or augmentation",
            "optimizer": "Adam, default betas/eps, no weight decay",
            "dataset": "MNIST",
            "split": "train [0:10000], val [50000:55000], test all 10000",
            "seeds": [int(x) for x in SEEDS],
            "lr_candidates": [float(x) for x in LR_CANDIDATES],
            "demo": bool(demo),
            "torch_version": torch.__version__,
            "device": str(DEVICE),
        },
        "metrics": metrics_dict,
        "early_stop": {
            "epoch": best_epoch,
            "val_acc_noisy": float(val_noisy[best_epoch - 1]),
            "test_acc_at_stop": float(test_acc[best_epoch - 1]),
            "test_acc_last10_mean": float(test_acc[-10:].mean()),
            "max_test_acc": float(test_acc.max()),
            "optimal_test_epoch": optimal_test_epoch,
            "gap_to_optimal_test_acc": float(test_acc.max() - test_acc[best_epoch - 1]),
        },
        "wall_time_s": float(time.perf_counter() - t0),
    }


## 4. Training runs for the systematic condition

The systematic condition is trained with the same four learning rates and the same three seeds as NB1: 4 × 3 = 12 runs, each of 200 epochs. Together with NB1's 24 runs, this completes the project's 36-run grid. The learning rate is then selected in the same way as in NB1, by the highest mean early-stopping noisy-validation accuracy over the three seeds (section 5).

This section has three parts:

- **4.1 Cache helpers.** Each run is saved as one JSON file, `results/{condition}_lr{lr}_seed{seed}.json`. The naming is the same as in NB1, so runs from both notebooks are read in the same way.
- **4.2 Systematic runs.** With `QUICK = True`, the 12 cached runs are loaded. If the cache is incomplete, raise Error. With `QUICK = False`, each run is trained and saved; a run whose file already exists is loaded instead of retrained, so an interrupted session can resume. Because every run is fixed by its seed, loading a finished run gives the same result as retraining it.
- **4.3 NB1 results.** The 24 clean and random runs are loaded from NB1's cache, so the three conditions can be compared in the following sections.


### 4.1 Cache helpers

Three small functions: `result_path` gives the file name of a run, `save_result` writes a run to that file, and `load_result` reads it back.


In [31]:
def result_path(condition, lr, seed):

    file_name = f"{condition}_lr{lr:g}_seed{seed}.json"  # combine the parameter to create a name for each run
    
    return RESULTS_DIR / file_name  # save in the results folder

def save_result(result):

    cfg = result["config"]  # get the configuration from dictionary
    path = result_path(cfg["condition"], cfg["lr"], cfg["seed"]) # get path name using function result_path
    path.write_text(json.dumps(result, indent = 2))
    
    return path

def load_result(condition, lr, seed):

    path = result_path(condition, lr, seed)  # get the path using the function

    return json.loads(path.read_text())  # file back to json text then back to dict


### 4.2 Systematic runs

**4.2a** lists the 12 result files for the systematic condition and checks how many are already cached. **4.2b** then loads or trains, depending on `QUICK` and on the cache.


In [28]:
CONDITIONS_NB2 = ["systematic"] # make it the same structure as NB1

# 12 result files: one condition * four learning rates * three seeds.
# store in the list SYS_CATHE_PATHS
SYS_CACHE_PATHS = []

for condition in CONDITIONS_NB2:
    for lr in LR_CANDIDATES:
        for seed in SEEDS: 
            SYS_CACHE_PATHS.append(result_path(condition, lr, seed))

# check if the 12 files are exist
flags = []
for i in SYS_CACHE_PATHS:
    flags.append(i.exists())
SYS_CACHE_COMPLETE = all(flags)  # True if all Trues


#### 4.2b Load or train

**Goal.** This cell ends with `sys_results`, a list of the 12 systematic run dictionaries, in the format that `train_one_run` returns. Every later section reads from this list.

| Case | When | What happens |
|---|---|---|
| 1 | `QUICK = True` and all 12 files are cached | load the 12 runs |
| 2 | `QUICK = True` and the cache is incomplete | stop with an error that says to set `QUICK = False` |
| 3 | `QUICK = False` | for each run: load it if its file exists, otherwise train for 200 epochs and save it |

In [30]:
sys_results = []

if QUICK and SYS_CACHE_COMPLETE:
    # case 1, every systematic run is cacthed, so load all
    for condition in CONDITIONS_NB2:
        for lr in LR_CANDIDATES:
            for seed in SEEDS:
                sys_results.append(load_result(condition, lr, seed))
    print(f"loaded {len(sys_results)} cached systematic runs")

elif QUICK:
    # case 2, quick is true but some runs are not exist, stop and raise an error
    raise FileNotFoundError(
        f"only {sum(flags)} / {len(SYS_CACHE_PATHS)} systematic runs are cached;"
        "set QUICK to false to train"
    )

else:
    # case 3, train
    for condition in CONDITIONS_NB2:
        for lr in LR_CANDIDATES:
            for seed in SEEDS:
                print(f"\n {condition} | lr={lr:g} | seed={seed}")
                if result_path(condition, lr, seed).exists():
                    # if already exist, no need to retrain, load it
                    print("cached, loading..")
                    result = load_result(condition, lr, seed)
                else:
                    result = train_one_run(condition, lr, seed, data)
                    save_result(result)
                sys_results.append(result)
    print(f"\n systematic runs complete: {len(sys_results)}")

FileNotFoundError: only 0 / 12 systematic runs are cached;set QUICK to false to train